In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fredapi import Fred

print("All packages imported successfully!")

All packages imported successfully!


In [26]:
df = pd.read_excel(r"C:\Users\adamp\Downloads\Treasury yields.xlsx")
print(df.head())
df['Spread'] = df['Y10'] - df['Y2']

                  date    Y2   Y10  Unnamed: 3
0  2020-02-01 00:00:00  1.58  1.88         NaN
1  2020-03-01 00:00:00  1.53  1.80         NaN
2  2020-06-01 00:00:00  1.54  1.81         NaN
3  2020-07-01 00:00:00  1.54  1.83         NaN
4  2020-08-01 00:00:00  1.58  1.87         NaN


In [16]:
#Spreadt​=Y10-Y2

#logic:
#Mean-Reversion (Z-Score)
#Compute rolling mean & std (e.g., 2 years ≈ 504 trading days)

#Go steepener when z-score < −1

#Go flattener when z-score > +1

#Exit when z-score crosses 0

#position encoding
#+1 = Long 10Y / Short 2Y  (enter steepener)
#-1 = Short 10Y / Long 2Y  (enter flattener)
#0 = Flat


In [29]:
import pandas as pd

# df must already contain columns: 'Y2', 'Y10'
df['Spread'] = df['Y10'] - df['Y2']

# Rolling stats (2-year window)
window = 504
df['SpreadMean'] = df['Spread'].rolling(window).mean()
df['SpreadStd'] = df['Spread'].rolling(window).std()

# Z-score
df['Z'] = (df['Spread'] - df['SpreadMean']) / df['SpreadStd']

# Initialize position
df['Position'] = 0

# Entry signals
df.loc[df['Z'] < -1.0, 'Position'] = 1
df.loc[df['Z'] >  1.0, 'Position'] = -1

# Exit rule: flatten when Z crosses 0
df['Position'] = df['Position'].where(df['Z'].abs() >= 0.0, 0)

# Forward-fill position to hold until exit
df['Position'] = df['Position'].ffill().fillna(0)

df[['Spread', 'Z', 'Position']].dropna().tail(400)


,Spread,Z,Position
1100,-0.44,0.108273,0
1101,-0.47,0.003008,0
1102,-0.40,0.267634,0
1103,-0.35,0.460579,0
1104,-0.37,0.393311,0
...,...,...,...
1495,0.41,1.661661,-1
1496,0.40,1.616187,-1
1497,0.40,1.610936,-1
1498,0.42,1.685722,-1
